# Lesson 02 - A Microsoft Agent Framework felfedezése

A **Microsoft Agent Framework (MAF)** egy egységes keretrendszer AI ügynökök létrehozásához. Tiszta, összerakható architektúrát kínál négy alapvető építőelemmel:

- **Client** – kapcsolódik egy AI modell végponthoz és kezeli a kommunikációt
- **Agent** – becsomagolja az ügyfelet utasításokkal és eszközdefiníciókkal
- **Tools** – kiterjesztik az ügynök képességeit egyedi függvényekkel, amelyeket a modell hívhat meg
- **Session** – megőrzi a beszélgetés előzményeit több körös interakciókhoz

Ebben a leckében létrehozunk egy **utazásfoglaló ügynököt**, amely a célpont elérhetőségét ellenőrzi ezekkel a fogalmakkal.


## Beállítás


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Az Agent Framework architektúrájának megértése

A Microsoft Agent Framework réteges architektúrát követ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Ügyfél** – Egy `AzureAIProjectAgentProvider` csatlakozik egy Azure OpenAI telepítéshez. Ez kezeli az azonosítást, a kérés formázását és a válasz feldolgozását.
2. **Agent** – Az ügyféltől a `provider.create_agent()` segítségével létrehozott agent kombinálja a modellhez való hozzáférést az utasításokkal (rendszer prompt) és az eszközökkel.
3. **Eszközök** – Python függvények, amelyeket `@tool` dekorátorral látnak el, és amelyeket az agent hívhat meg műveletek végrehajtására vagy adatok lekérésére.
4. **Munkamenet** – Egy `AgentSession` objektum (az `agent.create_session()` által létrehozott), amely tárolja a beszélgetés előzményeit, lehetővé téve a többszörös fordulós párbeszédet, ahol az agent emlékszik a korábbi kontextusra.

Építsük fel lépésről lépésre az egyes rétegeket.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Eszközök hozzáadása az @tool dekorátorral

Az eszközök lehetővé teszik az ügynökök számára, hogy szöveg generálásán túl is végrehajtsanak műveleteket. A `@tool` dekorátor egy normál Python függvényt átalakít olyanná, amelyet az ügynök meghívhat.

Főbb pontok:
- Használd az `Annotated[type, "description"]` formátumot, hogy a modell megértse az egyes paramétereket.
- A docstring lesz az eszköz leírása, amelyet a modell lát.
- Az `approval_mode="never_require"` azt jelenti, hogy az eszköz automatikusan lefut, felhasználói megerősítés nélkül.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Ügynök létrehozása eszközökkel

Most összevonjuk az ügyfelet, az utasításokat és az eszközöket egy ügynökbe. Az `instructions` a rendszerparancsként működik — meghatározzák az ügynök személyiségét és viselkedését.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Többkörös beszélgetések munkamenetekkel

Egy `AgentSession` (az `agent.create_session()` segítségével létrehozva) nyomon követi az összes üzenetet egy beszélgetésben. Azáltal, hogy ugyanazt a munkamenetet adjuk át minden `agent.run()` hívásnak, az ügynök hozzáfér a teljes beszélgetési előzményekhez, és visszautalhat korábbi üzenetekre.

Átadjuk a `tools=[check_destination_availability]` paramétert, hogy az ügynök minden körben használhassa az elérhetőség-ellenőrzőnket.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Összefoglaló

Ebben a leckében felfedezted a Microsoft Agent Framework négy pillérét:

| Fogalom | Amit megtanultál |
|---------|------------------|
| **Ügyfél** | Az `AzureAIProjectAgentProvider` hitelesítés-alapú hozzáféréssel kapcsolódik az Azure OpenAI-hoz |
| **Ügynök** | A `provider.create_agent()` egy modellikapcsolatot csomagol utasításokkal és névvel |
| **Eszközök** | Az `@tool` dekorátor Python funkciókat tesz elérhetővé az ügynök számára hívásra |
| **Munkamenet** | Az `agent.create_session()` több fordulón át megőrzi a beszélgetés előzményeit |

Ezek az építőelemek együtt alkotnak olyan ügynököket, amelyek természetes beszélgetéseket tudnak folytatni, külső funkciókat hívhatnak meg, és képesek megőrizni a kontextust — ez az alapja a későbbi leckék fejlettebb ügynöki mintáinak.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:  
Ez a dokumentum az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hivatalos forrásnak. Kritikus információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
